# 03. 접수 수요 시간대/요일 분석

접수일시 기준 시간대별, 요일별, 요일 X 시간대별 요청 수요를 분석한다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

df_request = pd.read_csv(
    'data/서울시설공단_장애인콜택시 접수일시_월주차_파생컬럼_20251231.csv',
    parse_dates=['접수일시', '접수일자']
)

df_request.info()
df_request.head()


### 시간대별 접수 수요

In [ ]:
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

hourly_request_count = (
    df_request
    .dropna(subset=['접수시간대'])
    .groupby('접수시간대')
    .size()
    .reindex(range(24), fill_value=0)
    .reset_index(name='접수건수')
)

hourly_request_count['시간대'] = hourly_request_count['접수시간대'].map(
    lambda x: f'{int(x):02d}시'
)

display(hourly_request_count)

In [ ]:
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

hourly_request_count = (
    df_request
    .dropna(subset=['접수시간대'])
    .groupby('접수시간대')
    .size()
    .reindex(range(24), fill_value=0)
    .reset_index(name='접수건수')
)

hourly_request_count['시간대'] = hourly_request_count['접수시간대'].map(
    lambda x: f'{int(x):02d}시'
)

hourly_heatmap_data = hourly_request_count.set_index('시간대')[['접수건수']]

fig, (ax_heatmap, ax_bar) = plt.subplots(
    1,
    2,
    figsize=(16, 10),
    gridspec_kw={'width_ratios': [1, 2.2]}
)

sns.heatmap(
    hourly_heatmap_data,
    ax=ax_heatmap,
    cmap='YlOrRd',
    annot=True,
    fmt=',.0f',
    linewidths=0.5,
    cbar=False
)

ax_heatmap.set_title('시간대별 접수건수 히트맵')
ax_heatmap.set_xlabel('접수 지표')
ax_heatmap.set_ylabel('시간대')
ax_heatmap.tick_params(axis='y', rotation=0)

ax_bar.barh(
    hourly_request_count['시간대'],
    hourly_request_count['접수건수'],
    color='#4c78a8'
)

ax_bar.invert_yaxis()
ax_bar.set_title('시간대별 접수건수')
ax_bar.set_xlabel('접수건수')
ax_bar.set_ylabel('시간대')
ax_bar.grid(axis='x', alpha=0.3)

for index, value in enumerate(hourly_request_count['접수건수']):
    ax_bar.text(
        value,
        index,
        f' {value:,.0f}',
        va='center'
    )

plt.tight_layout()
plt.show()

### 요일별 접수 수요

In [ ]:
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

weekday_order = ['월요일', '화요일', '수요일', '목요일', '금요일', '토요일', '일요일']

weekday_request_count = (
    df_request
    .dropna(subset=['접수요일'])
    .groupby('접수요일')
    .size()
    .reindex(weekday_order, fill_value=0)
    .reset_index(name='접수건수')
)

weekday_request_count['접수비중(%)'] = (
    weekday_request_count['접수건수']
    / weekday_request_count['접수건수'].sum()
    * 100
)

display(weekday_request_count)

In [ ]:
weekday_heatmap_data = weekday_request_count.set_index('접수요일')[['접수건수']]

fig, (ax_heatmap, ax_bar) = plt.subplots(
    1,
    2,
    figsize=(14, 7),
    gridspec_kw={'width_ratios': [1, 2.2]}
)

sns.heatmap(
    weekday_heatmap_data,
    ax=ax_heatmap,
    cmap='YlOrRd',
    annot=True,
    fmt=',.0f',
    linewidths=0.5,
    cbar=False
)

ax_heatmap.set_title('요일별 접수건수 히트맵')
ax_heatmap.set_xlabel('접수 지표')
ax_heatmap.set_ylabel('요일')
ax_heatmap.tick_params(axis='y', rotation=0)

ax_bar.barh(
    weekday_request_count['접수요일'],
    weekday_request_count['접수건수'],
    color='#4c78a8'
)

ax_bar.invert_yaxis()
ax_bar.set_title('요일별 접수건수')
ax_bar.set_xlabel('접수건수')
ax_bar.set_ylabel('요일')
ax_bar.grid(axis='x', alpha=0.3)

for index, value in enumerate(weekday_request_count['접수건수']):
    rate = weekday_request_count.loc[index, '접수비중(%)']
    ax_bar.text(
        value,
        index,
        f' {value:,.0f}건 ({rate:.1f}%)',
        va='center'
    )

plt.tight_layout()
plt.show()

### 요일 X 시간대별 접수 수요

In [ ]:
weekday_order = ['월요일', '화요일', '수요일', '목요일', '금요일', '토요일', '일요일']

request_weekday_hour_pivot = (
    df_request
    .dropna(subset=['접수요일', '접수시간대'])
    .pivot_table(
        index='접수요일',
        columns='접수시간대',
        values='접수일시',
        aggfunc='count',
        fill_value=0
    )
    .reindex(weekday_order)
)

request_weekday_hour_pivot.columns = [
    f'{int(hour):02d}시' for hour in request_weekday_hour_pivot.columns
]

request_weekday_hour_pivot

In [ ]:
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(18, 7))

sns.heatmap(
    request_weekday_hour_pivot,
    cmap='YlOrRd',
    annot=True,
    fmt=',.0f',
    linewidths=0.5,
    cbar_kws={'label': '접수건수'}
)

plt.title('요일 X 시간대별 접수 수요 히트맵')
plt.xlabel('접수시간대')
plt.ylabel('접수요일')
plt.xticks(rotation=0)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()